# E9-E11 — Split, Naive, and Weighted Conformal Prediction (Gate 2)

**Experiment IDs:** `E9` (machinery validation), `E10` (naive coverage failure), `E11` (weighted
correction — **Gate 2**). **Spec:** `EXPERIMENT_PLAN.md`. **Governing rules:** `CLAUDE.md`.

**Contribution 1** — coverage-valid intervals on the selection-biased official test set via weighted
conformal prediction. This notebook establishes the machinery is correct (E9), demonstrates the
problem (E10), and delivers the correction (E11).

**Design-review resolutions in force** (`DECISIONS.md`, 2026-09-15):
- **Q-CONF-01** — one nonconformity score per event per horizon, calibrated on the full training pool.
- **Q-SEL-01** — rule-derived likelihood-ratio weights PRIMARY; classifier-estimated weights a
  SECONDARY robustness check, agreement quantified by γ̂. Exact finite-sample coverage is claimed
  ONLY for the rule-derived weights.
- **Q-SEL-03** — mandatory diagnostics (n, n̂, Pareto k̂, weight summary, ASMD on the E1-flagged
  covariates); conditional clipping only if k̂ > 0.7; supported/unsupported positivity partition.
- **Q-STAT-04** — ONE pre-registered formal contrast: naive (E10) vs weighted (E11) marginal
  coverage on the official test set, primary level, two-sided. Everything else is descriptive.

**Base learners (2026-09-01 amendment):** persistence (E5 LRP), GBM (E6), GRU (E7), MC-dropout (E8).

**Scope discipline:** the official test set is read once per experiment for scoring only; all model
selection reused the Phase-2 cached searches. This notebook STOPS at Gate 2 — the GO/PIVOT/NO-GO
call is Sidh's.

**Implementation deviation flagged (Q-SEL-03):** the Pareto k̂ is the GPD shape parameter estimated
by scipy's MLE fit rather than the Zhang–Stephens PSIS variant the resolution names; same shape
parameter, same 0.5/0.7/1.0 bands, vetted estimator over a hand-rolled one.

In [ ]:
# --- Setup, configuration, provenance (invariant I4) ----------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import conformal_runner as R

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

from kelvins_conformal.reporting import write_table_atomic
def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{name}.csv")
    print(f"saved: reports/tables/{name}.csv")

PROVENANCE = {
    "experiment_ids": ["E9", "E10", "E11"],
    "git_commit_sha": git_sha(), "config_hash": cfg.config_hash, "seed": cfg.seed,
    "seeds": list(cfg.train.seeds), "bootstrap_resamples": cfg.bootstrap.n_resamples,
    "nominal_levels": [cfg.power.nominal_coverage_primary, *cfg.power.nominal_coverage_secondary],
    "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0],
}
print(json.dumps(PROVENANCE, indent=2))
CPERS, CGBM, CGRU, CBAY = "#000000", "#0072B2", "#D55E00", "#009E73"

## 1. Run E9-E11

The heavy step: fits the four base learners (persistence is free) on the fit split, calibrates on
the `calibration` subset, and evaluates split conformal on the self-test split (E9) and official
test (E10), then weighted conformal on the supported official-test region (E11), across 3 seeds ×
2 sidedness × 3 nominal levels, with event-level bootstrap CIs.

In [ ]:
RES = R.run_all(cfg)
meta = RES["meta"]
print("seeds:", meta["seeds"], "| levels:", meta["levels"], "| primary:", meta["primary_level"])
print(f"train high-risk prevalence: {meta['train_prev']:.4f}  "
      f"test high-risk prevalence: {meta['test_prev']:.4f}")
for key in ("coverage", "diagnostics", "weights", "primary_contrast", "positivity"):
    save_table(RES[key], f"e9e11_{key}")

## 2. Weight construction and its diagnostics (Q-SEL-01, Q-SEL-03)

Before any coverage claim, the weights themselves are audited: agreement between the rule-derived
(primary) and classifier-estimated (secondary) constructions, effective sample size, and tail
shape.

In [ ]:
w = RES["weights"].iloc[0]
print(f"γ̂ (max multiplicative divergence, rule vs classifier): {w['gamma_divergence']:.3f}")
print(f"discriminator AUC (calibration vs official test):       {w['classifier_auc']:.3f}")
print(f"rule weights:       n={int(w['rule_n'])}  n_eff={w['rule_n_effective']:.1f}  "
      f"k̂={w['rule_khat']:.3f}")
print(f"classifier weights: n_eff={w['classifier_n_effective']:.1f}  k̂={w['classifier_khat']:.3f}")
display(RES["weights"].T)

# Weight distribution figure (rule-derived).
rw = RES["rule_weights"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(rw, bins=40, color=CGBM, alpha=0.8)
ax[0].set_xlabel("rule-derived weight"); ax[0].set_ylabel("calibration events")
ax[0].set_title("(a) weight distribution")
pos = rw[rw > 0]
ax[1].hist(np.log10(pos + 1e-12), bins=40, color=CGRU, alpha=0.8)
ax[1].set_xlabel("log10 weight (positive only)"); ax[1].set_ylabel("events")
ax[1].set_title(f"(b) log-scale; {np.mean(rw==0):.0%} at zero (non-recency)")
fig.suptitle("E11 rule-derived calibration weights", y=1.02)
save_fig(fig, "e11_weight_distribution")
plt.show()

In [ ]:
# Positivity partition (Q-SEL-03 C).
pos = RES["positivity"].iloc[0]
print("POSITIVITY PARTITION (official test set):")
print(f"  total test events : {int(pos['n_test_total'])}")
print(f"  supported         : {int(pos['n_supported'])}")
print(f"  unsupported       : {int(pos['n_unsupported'])} "
      f"(high-risk among them: {int(pos['unsupported_high_risk'])})")
print("Coverage below is claimed on the SUPPORTED region; the unsupported subpopulation is")
print("reported here, not silently dropped (Q-SEL-03 C).")
display(RES["positivity"])

## 3. E9 — machinery validation on the exchangeable self-split

If split conformal is implemented correctly, coverage on a randomly-drawn (exchangeable) self-test
split must match nominal. A deviation here is an implementation bug, not selection bias — it would
block everything downstream (E9 failure criterion).

In [ ]:
cov = RES["coverage"]
def view(method, sided="two"):
    v = cov[(cov["method"] == method) & (cov["sided"] == sided)].copy()
    return v[["learner", "nominal", "coverage_mean", "coverage_sd", "gap_pp",
             "cp_lo_mean", "cp_hi_mean", "median_width_mean", "n"]].round(4)

e9 = view("E9_selftest")
display(e9)
save_table(e9, "e9_machinery_validation")
worst = e9.loc[e9["gap_pp"].abs().idxmax()]
print(f"largest |gap| from nominal on the self-split: {worst['gap_pp']:.2f} pp "
      f"({worst['learner']} @ {worst['nominal']:.0%})")
print("Expectation: coverage ~ nominal for every learner/level (exchangeable conditions).")

## 4. E10 — naive conformal on the official (biased) test set

Same calibration, evaluated on the selection-biased official test set. The hypothesis is
under-coverage — the problem Contribution 1 solves.

In [ ]:
e10 = view("E10_naive_official")
display(e10)
save_table(e10, "e10_naive_official")

# Figure: E9 (self-split) vs E10 (official) coverage at each nominal level — the problem.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, lrn in zip(axes, R.BASE_LEARNERS):
    for method, c, mk in (("E9_selftest", CGBM, "o"), ("E10_naive_official", CGRU, "s")):
        v = cov[(cov.method==method)&(cov.sided=="two")&(cov.learner==lrn)].sort_values("nominal")
        ax.errorbar(v["nominal"], v["coverage_mean"],
                    yerr=[v["coverage_mean"]-v["cp_lo_mean"], v["cp_hi_mean"]-v["coverage_mean"]],
                    marker=mk, color=c, capsize=3,
                    label=("E9 self-split" if method=="E9_selftest" else "E10 official"))
    ax.plot([0.78,0.97],[0.78,0.97], "k--", lw=1, alpha=0.6)
    ax.set_title(lrn); ax.set_xlabel("nominal"); ax.set_xlim(0.78,0.97)
axes[0].set_ylabel("empirical coverage"); axes[0].legend(fontsize=8, frameon=False)
fig.suptitle("E9 (exchangeable self-split) vs E10 (naive, official biased test) — the problem", y=1.03)
save_fig(fig, "e10_problem_selfsplit_vs_official")
plt.show()

## 5. E11 — weighted conformal (the correction) — Gate 2

Rule-derived weights (primary; entitled to the exact finite-sample claim) and classifier weights
(secondary robustness check). Full diagnostics reported per the mandatory set.

In [ ]:
e11r = view("E11_weighted_rule")
e11c = view("E11_weighted_classifier")
print("=== E11 weighted (rule-derived, PRIMARY) ==="); display(e11r)
print("=== E11 weighted (classifier-estimated, SECONDARY robustness) ==="); display(e11c)
save_table(e11r, "e11_weighted_rule"); save_table(e11c, "e11_weighted_classifier")
display(RES["diagnostics"].round(4))
save_table(RES["diagnostics"], "e11_weight_diagnostics")

In [ ]:
# Headline figure: naive (E10) vs weighted (E11 rule) coverage, official test.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, lrn in zip(axes, R.BASE_LEARNERS):
    for method, c, mk, lab in (("E10_naive_official", CGRU, "s", "E10 naive"),
                               ("E11_weighted_rule", CBAY, "D", "E11 weighted")):
        v = cov[(cov.method==method)&(cov.sided=="two")&(cov.learner==lrn)].sort_values("nominal")
        ax.errorbar(v["nominal"], v["coverage_mean"],
                    yerr=[v["coverage_mean"]-v["cp_lo_mean"], v["cp_hi_mean"]-v["coverage_mean"]],
                    marker=mk, color=c, capsize=3, label=lab)
    ax.plot([0.78,0.97],[0.78,0.97], "k--", lw=1, alpha=0.6)
    ax.set_title(lrn); ax.set_xlabel("nominal"); ax.set_xlim(0.78,0.97)
axes[0].set_ylabel("empirical coverage"); axes[0].legend(fontsize=8, frameon=False)
fig.suptitle("HEADLINE — E10 naive vs E11 weighted coverage on the official test set", y=1.03)
save_fig(fig, "e11_headline_naive_vs_weighted")
plt.show()

## 6. The pre-registered primary contrast (Q-STAT-04)

Exactly one formal test: naive (E10) vs weighted (E11 rule) marginal coverage on the official test
set, primary level, two-sided. Tested here with **McNemar's test** on the paired per-event coverage
indicators (the two methods are evaluated on the same events, so the pairing is exact). Everything
else in this notebook is descriptive with CIs.

**Primary base learner: persistence** — it is deterministic (no seed variance) and is the dataset's
documented strong baseline, so the contrast isolates the *weighting* effect rather than base-learner
noise. This choice within the Q-STAT-04 resolution is flagged for Sidh; the contrast is shown
descriptively for all four learners alongside.

In [ ]:
display(RES["primary_contrast"].round(4))
save_table(RES["primary_contrast"], "e11_primary_contrast")

# McNemar on paired coverage indicators, persistence, primary level, two-sided, seed 42.
prim = meta["primary_level"]; alpha = 1.0 - prim
events = R.load_events(cfg); data = R.prepare_conformal_data(cfg, events)
weights = R.build_weights(cfg, data)
preds = R.base_predictions(cfg, data, cfg.seed)
cal, test = data.subsets["calibration"], data.subsets["official_test"]
part = R.diag.positivity_partition(test["recency_ok"]); sup = part.supported
scores = R.make_scores(cal["y"], preds["persistence"]["calibration"], "two")

iv10 = R.split_interval(preds["persistence"]["official_test"][sup], scores, alpha, sided="two")
res11 = R.weighted_interval(preds["persistence"]["official_test"][sup], scores, weights.rule, alpha, sided="two")
c10 = iv10.covers(test["y"][sup]); c11 = res11.interval.covers(test["y"][sup])

# McNemar 2x2 on discordant pairs.
b = int(np.sum(c10 & ~c11))   # covered by naive, not weighted
c_ = int(np.sum(~c10 & c11))  # covered by weighted, not naive
from scipy.stats import binomtest
mcnemar_p = float(binomtest(min(b, c_), b + c_, 0.5).pvalue) if (b + c_) > 0 else 1.0
print(f"PRIMARY CONTRAST (persistence, nominal {prim:.0%}, two-sided, supported region):")
print(f"  E10 naive coverage    : {c10.mean():.4f}")
print(f"  E11 weighted coverage : {c11.mean():.4f}")
print(f"  discordant pairs: naive-only={b}, weighted-only={c_}")
print(f"  McNemar exact two-sided p = {mcnemar_p:.4g}")
save_table(pd.DataFrame([{"learner":"persistence","nominal":prim,
    "E10_coverage":float(c10.mean()),"E11_coverage":float(c11.mean()),
    "naive_only":b,"weighted_only":c_,"mcnemar_p":mcnemar_p}]), "e11_primary_mcnemar")

## 7. Gate 2 summary — measurement only, no go/no-go

Restating what the batch shows. **This notebook does not make the Gate 2 call**, does not confirm
Contribution 1 as validated, and does not proceed to E12/E13 — those are Sidh's (CLAUDE.md §3, §10,
§13).

In [ ]:
summ = []
def g(method, lrn, lvl, sided="two"):
    v = cov[(cov.method==method)&(cov.learner==lrn)&(cov.nominal==lvl)&(cov.sided==sided)]
    return float(v["coverage_mean"].iloc[0]), float(v["gap_pp"].iloc[0])
prim = meta["primary_level"]
for lrn in R.BASE_LEARNERS:
    c9,_ = g("E9_selftest", lrn, prim); c10,g10 = g("E10_naive_official", lrn, prim)
    c11,g11 = g("E11_weighted_rule", lrn, prim)
    summ.append({"learner":lrn, f"E9 self ({prim:.0%})":round(c9,3),
                 "E10 naive":round(c10,3), "E10 gap pp":round(g10,1),
                 "E11 weighted":round(c11,3), "E11 gap pp":round(g11,1),
                 "gap closed pp":round(100*(c11-c10),1)})
gate = pd.DataFrame(summ).set_index("learner")
display(gate); save_table(gate.reset_index(), "e9e11_gate2_summary")

(cfg.path("reports_dir")/"03_conformal_provenance.json").write_text(json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("""
E9-E11 / GATE 2 — WHAT THE BATCH SHOWS (measurement only)
 * E9: coverage on the exchangeable self-split (machinery validation).
 * E10: coverage on the official biased test set under naive conformal (the problem).
 * E11: coverage after rule-derived weighting (the correction), with full weight diagnostics,
   the positivity partition, and the classifier-weight robustness check.
 * Primary pre-registered contrast (persistence, naive vs weighted): McNemar test above.

NOT DECIDED HERE (CLAUDE.md §3, §10, §13.7):
 * the Gate 2 GO/PIVOT/NO-GO call and whether Contribution 1 is "validated";
 * confirmation of the persistence choice for THE primary contrast;
 * anything in E12/E13 or later — execution stops at this batch boundary.
""")
print("provenance:", cfg.path("reports_dir")/"03_conformal_provenance.json")